# Wake County Restaurant Inspections: Analysis Pipeline

*This notebook was generated by a language model from the prompt: "Analyze these two
restaurant inspection files and produce a county-wide report."*

**It runs top to bottom with zero errors. It is wrong in three places.**

This is the pipeline audited in **Lab 7, Task 2.2**. Run it, confirm it raises nothing,
then find as many of the three defects as possible. For each one: name the cell, state
what it does versus what it should do, and explain **why it raised no exception**.

In [ ]:
import pandas as pd

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users

inspections = pd.read_csv(DATA + "inspections_a.csv")
restaurants = pd.read_csv(DATA + "inspections_b.csv")

print(f"inspections loaded          : {len(inspections):,} rows")
print(f"restaurant reference loaded : {len(restaurants):,} rows")

In [ ]:
# Enrich each inspection with the restaurant's attributes (name, city, cuisine).
df = inspections.merge(restaurants, on="restaurant_id", how="left")

print(f"enriched dataset: {len(df):,} rows")
print("left join used — no inspections were lost in the merge.")

In [ ]:
# Rank every inspection from worst to best, then list the ten worst county-wide.
worst_first = df.sort_values("score")
df["score_rank"] = worst_first["score"].rank().values

ten_worst = df.nsmallest(10, "score_rank")

print("TEN WORST INSPECTIONS, COUNTY-WIDE")
print("-" * 60)
print(ten_worst[["restaurant_name", "city", "score", "score_rank"]]
      .to_string(index=False))

In [ ]:
# County summary tables.
by_city = df.groupby("city")["score"].agg(["mean", "min", "count"]).round(2)

below_85 = int((df["score"] < 85).sum())
reinspection_rate = (df["reinspection_required"] == "Y").mean() * 100

print("MEAN SCORE BY CITY")
print("-" * 40)
print(by_city.to_string())
print()
print(f"inspections scoring below 85 : {below_85}")
print(f"reinspection rate            : {reinspection_rate:.1f}%")

In [ ]:
# Data-quality check: certify one row per inspection before publishing.
df = df.drop_duplicates("inspection_id")
assert df["inspection_id"].is_unique, "duplicate inspections found"

print(f"final dataset: {len(df):,} unique inspections")
print("Quality check passed. Report certified against 1,200 unique inspections.")

**Report complete.** All cells ran without error and the final quality check passed.

*(That last sentence is true. Lab 7 asks why it is also worthless.)*